# Camera-ready IID experiments — fixed LRao + amplitude sweep
**Upload-and-run (GPU recommended).** Reruns the paper's IID experiments
(Pavia single-class + multi-class, plus San Diego I-II in multiclass mode:
random non-aircraft pixels as background, aircraft mean spectrum as the
signature; original `src/iid.py` code and configs)
with the **fixed LRao** (robust median/IQR input normalization; DART keeps its
ZCA whitening; AMF is the pure unregularized version, as in the original path).
Then an **amplitude sweep** at the largest training size, reusing the trained
checkpoints: θ ∈ {0.03, 0.075, 0.15, 0.225, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95}.
Outputs per mode: the paper figures (AUC/pAUC/Pd vs n, vs ρ, ROCs), all trained
models, metrics + raw scores, the new `auc_vs_theta` / `pd_at_fa_vs_theta`
figures, and per-(seed, θ) raw score archives. Everything zips + downloads at
the end. Cells are independent — a disconnect loses at most one cell.
Runtime: each run_iid cell is hours (5 seeds × 2000-epoch nets × n- and ρ-sweeps);
the θ-sweep itself is minutes (scoring only, plus per-θ GMM-Levin fits in multi).


In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, torch, yaml
sys.path.insert(0, '.')
import tsp_repro  # path shim -> vendored original src/
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
# ---- FIXED LRao: robust (median/IQR) normalization for the LRao nets ----
from tsp_repro import iid_camera_ready as CR
CR.apply_robust_lrao()
print('theta grid:', CR.THETAS)


In [ ]:
# ---- configs: original YAMLs + the notebook's original overrides ----
ALG_IID = dict(dsm_sigma_rho=0.1, whiten_eig_floor=0.0,
               lfi_delta_theta=0.01, lfi_detach_sigma=True)
ALG_IID_SINGLE = dict(ALG_IID, gmm_K=2, lfi_sigma_cutoff=1e-6, lrao_whiten_eig_floor=1e-10)
ALG_IID_MULTI  = dict(ALG_IID, gmm_K=9, lfi_sigma_cutoff=1e-3, lrao_whiten_eig_floor=1e-5)

def load_cfg(path, overrides):
    cfg = yaml.safe_load(open(path))
    cfg['device'] = DEVICE
    cfg['dataset'] = 'colab_deep/data/pavia-u.mat'   # bundled on this branch
    cfg.update(overrides)
    return cfg

cfg_single = load_cfg('tsp_repro/configs/iid_single.yaml', ALG_IID_SINGLE)
cfg_multi  = load_cfg('tsp_repro/configs/iid_multi.yaml',  ALG_IID_MULTI)
print('single:', {k: cfg_single[k] for k in ('bkg_cls','target_cls','n_train_list','seed')})
print('multi :', {k: cfg_multi[k]  for k in ('target_cls','exclude_classes','n_train_list','seed')})


## Pavia — single-class IID (5 seeds; hours)


In [ ]:
from src.iid import run_iid
agg_single, m_single = run_iid(cfg_single, mode='single')
print('single aggregate dir:', agg_single)


## Pavia — multi-class IID (5 seeds; hours)


In [ ]:
agg_multi, m_multi = run_iid(cfg_multi, mode='multi')
print('multi aggregate dir:', agg_multi)


## Amplitude sweep (reuses the n=2000 checkpoints)
Now ALSO trains the four deep baselines (THANTD, HTD-Net, TSTTD, OS-VAE) per
seed on the SAME 2000 training pixels + signature, each with its own paper
recipe (checkpoint-resumable under theta_sweep/ckpt_deep), and scores them on
the same planted sets. GPU strongly recommended for this block
(~3-5 min per (deep model, seed): 4 models x 5 seeds per dataset).


In [ ]:
res_th_single = CR.theta_sweep(agg_single, cfg_single, 'single', deep=CR.DEEP_BASELINES)



In [ ]:
res_th_multi = CR.theta_sweep(agg_multi, cfg_multi, 'multi', deep=CR.DEEP_BASELINES)



## San Diego I--II — multiclass IID (no labels -> random background pixels)
Background pool = all non-aircraft pixels (9,942 / 9,866); signature = mean
spectrum of the real aircraft (58 / 134 pixels). Same protocol, thetas, and
detector set (incl. GMM-Levin) as Pavia multiclass.


In [ ]:
def sd_cfg(mat, results_dir):
    cfg = dict(cfg_multi)                 # same protocol as Pavia multiclass
    cfg.update(dataset=mat, target_cls=1, exclude_classes=[],
               results_dir=results_dir)
    return cfg

cfg_sd1 = sd_cfg('tsp_repro/data/Sandiego.mat',  'results/iid_sd1')
cfg_sd2 = sd_cfg('tsp_repro/data/Sandiego2.mat', 'results/iid_sd2')


In [ ]:
agg_sd1, m_sd1 = run_iid(cfg_sd1, mode='multi')
print('SD1 aggregate dir:', agg_sd1)


In [ ]:
agg_sd2, m_sd2 = run_iid(cfg_sd2, mode='multi')
print('SD2 aggregate dir:', agg_sd2)


In [ ]:
res_th_sd1 = CR.theta_sweep(agg_sd1, cfg_sd1, 'multi', deep=CR.DEEP_BASELINES)



In [ ]:
res_th_sd2 = CR.theta_sweep(agg_sd2, cfg_sd2, 'multi', deep=CR.DEEP_BASELINES)



## Show the key figures inline


In [ ]:
import glob
from IPython.display import Image, display
for d in (agg_single, agg_multi, agg_sd1, agg_sd2):
    for fig in ('figures/pd_at_fa_vs_n.png', 'figures/pdet_at_pfa_vs_rho.png',
                'theta_sweep/figures/auc_vs_theta.png'):
        p = glob.glob(f'{d}/{fig}')
        if p: print(p[0]); display(Image(p[0], width=640))



## Zip everything + download


In [ ]:
CR.zip_and_download([agg_single, agg_multi, agg_sd1, agg_sd2], zip_name='iid_camera_ready.zip')

